## TRansform into global brain atlas
Reads df_meta_with_clusters.tsv from the same directory as the script (Option 1).

Collapses it to contact-level (df_meta_contact_level.tsv) so (patient_id, electrode) is unique.

Loads each patient’s *_contacts_tkrRAS.csv from your Paper1_recons outputs.

Transforms subject tkrRAS → fsaverage tkrRAS using:

each subject’s mri/transforms/talairach.xfm

fsaverage’s mri/transforms/talairach.xfm

vox2ras and vox2ras_tkr from an MGZ header (brainmask/T1/orig)

Writes:

electrode_coords_fsaverage_tkr.tsv

In [3]:
#!/usr/bin/env python3
"""
00_prepare_and_transform_to_fsaverage.py

Creates (next to this script):
  1) df_meta_contact_level.tsv
  2) electrode_coords_fsaverage_tkr.tsv
  3) transform_qc_summary.tsv

Inputs:
  - df_meta_with_clusters.tsv  (must be in the same folder as this script)
  - Per-patient contact CSVs (subject tkrRAS) from your recon outputs:
      OUTPUTS_ROOT/<pid>/glassbrain/coords/<pid>_contacts_tkrRAS.csv

Required FreeSurfer files:
  - For each subject:
      <subj_dir>/mri/transforms/talairach.xfm
      <subj_dir>/mri/brainmask.mgz (or T1.mgz or orig.mgz)
  - For fsaverage:
      FSAVERAGE_DIR/mri/transforms/talairach.xfm
      FSAVERAGE_DIR/mri/brainmask.mgz (or T1.mgz or orig.mgz)

Transform chain (affine):
  subject tkrRAS -> subject scannerRAS -> MNI305 -> fsaverage scannerRAS -> fsaverage tkrRAS
"""

from __future__ import annotations

import re
from pathlib import Path

from pathlib import Path

def get_base_dir() -> Path:
    # .py -> script directory; Jupyter -> current working directory
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()

from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import nibabel as nib


# =============================================================================
# CONFIG
# =============================================================================

# -------- Meta (script-relative) --------
META_IN = "df_meta_with_clusters.tsv"
META_OUT = "df_meta_contact_level.tsv"

CLUSTER_COL = "cluster_kmeans_blob_weighted_bestK"
CLUSTER_COL = "cluster_kmeans_umap10d_bestK"
COLLAPSE_CLUSTER_STRATEGY = "mode"   # "mode" or "first"
FILTER_HIGH_ACTIVITY = False
HIGH_ACTIVITY_COL = "high_activity"

# -------- Patients --------
PATIENT_IDS: List[str] = [
    "PAT_2868","PAT_3066","PAT_3301","PAT_3390","PAT_3415","PAT_3455","PAT_3965","PAT_3975","PAT_3780",
    "MicroEPI-G-01","MicroEPI-G-02","MicroEPI-G-03","MicroEPI-G-04","MicroEPI-G-05"
]

# -------- Where your per-patient contact CSVs live (subject tkrRAS) --------
OUTPUTS_ROOT = r"\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\01_FBM_Analysis\outputs\Paper1_recons"
BLOCK_NAME = "glassbrain"
CSV_PRODUCT = "coords"

# -------- FreeSurfer subject dirs --------
ROOT_PAT   = r"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_HUG"
ROOT_MICRO = r"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\MICROEPI"

# fsaverage FreeSurfer directory (YOU HAVE THIS ALREADY)
FSAVERAGE_DIR = r"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_HUG\fsaverage"
# (If you prefer mapped drive S:, you can replace with:
# FSAVERAGE_DIR = r"S:\HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_HUG\fsaverage"
# )


# -------- Outputs root (NOT current directory) --------
OUT_ROOT = r"\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\atlas_inputs"
# (create if missing)

META_OUT = "df_meta_contact_level.tsv"
COORDS_OUT = "electrode_coords_fsaverage_tkr.tsv"
QC_OUT = "transform_qc_summary.tsv"


# =============================================================================
# UTIL
# =============================================================================

def read_table_auto(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=None, engine="python")
    df.columns = [c.strip() for c in df.columns]
    return df


def mode_int(series: pd.Series) -> Optional[int]:
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return None
    m = s.mode()
    return int(m.iloc[0]) if len(m) else int(s.iloc[0])


def collapse_meta_to_contact_level(df_meta: pd.DataFrame) -> pd.DataFrame:
    required = {"patient_id", "electrode", CLUSTER_COL}
    missing = required - set(df_meta.columns)
    if missing:
        raise ValueError(f"df_meta missing required columns: {sorted(missing)}")

    df = df_meta.copy()

    if FILTER_HIGH_ACTIVITY and HIGH_ACTIVITY_COL in df.columns:
        df = df.loc[df[HIGH_ACTIVITY_COL].astype(bool)].copy()

    df["patient_id"] = df["patient_id"].astype(str).str.strip()
    df["electrode"] = df["electrode"].astype(str).str.strip()

    if COLLAPSE_CLUSTER_STRATEGY == "mode":
        cluster_agg = (CLUSTER_COL, mode_int)
    elif COLLAPSE_CLUSTER_STRATEGY == "first":
        cluster_agg = (CLUSTER_COL, lambda s: int(pd.to_numeric(s, errors="coerce").dropna().iloc[0]))
    else:
        raise ValueError("COLLAPSE_CLUSTER_STRATEGY must be 'mode' or 'first'.")

    agg: Dict[str, tuple] = {"cluster": cluster_agg}
    if "condition" in df.columns:
        agg["conditions"] = ("condition", lambda s: ",".join(sorted(set(map(str, s)))))
    if "task" in df.columns:
        agg["tasks"] = ("task", lambda s: ",".join(sorted(set(map(str, s)))))
    if HIGH_ACTIVITY_COL in df.columns:
        agg["high_activity_any"] = (HIGH_ACTIVITY_COL, "max")

    out = df.groupby(["patient_id", "electrode"], as_index=False).agg(**agg)
    out = out.dropna(subset=["cluster"]).copy()
    out["cluster"] = out["cluster"].astype(int)

    dup = out.duplicated(subset=["patient_id", "electrode"]).sum()
    if dup:
        raise RuntimeError(f"Contact-level meta still has duplicates: {dup}")
    return out


def patient_freesurfer_dir(pid: str) -> Path:
    pid = str(pid)
    if pid.startswith("PAT_"):
        return Path(ROOT_PAT) / pid / "anatomy" / "prep" / "freesurfer"
    if pid.startswith("MicroEPI"):
        return Path(ROOT_MICRO) / pid / "anatomy" / "prep"
    raise ValueError(f"Unrecognized patient id pattern: {pid}")


def patient_contacts_csv(pid: str) -> Path:
    return Path(OUTPUTS_ROOT) / pid / BLOCK_NAME / CSV_PRODUCT / f"{pid}_contacts_tkrRAS.csv"


def load_patient_contacts(pid: str) -> pd.DataFrame:
    p = patient_contacts_csv(pid)
    if not p.exists():
        raise FileNotFoundError(f"Missing contacts CSV: {p}")

    df = pd.read_csv(p)
    df.columns = [c.strip() for c in df.columns]

    if "name" not in df.columns:
        raise ValueError(f"{pid}: contacts file missing 'name' column: {p}")
    for c in ["x", "y", "z"]:
        if c not in df.columns:
            raise ValueError(f"{pid}: contacts file missing '{c}' column: {p}")

    df = df.copy()
    df["patient_id"] = pid
    df = df.rename(columns={"name": "electrode"})
    df["electrode"] = df["electrode"].astype(str).str.strip()

    for c in ["x", "y", "z"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["x", "y", "z"]).copy()
    return df


# =============================================================================
# TRANSFORMS
# =============================================================================

def load_mgz_matrices(subj_dir: Path) -> Tuple[np.ndarray, np.ndarray, Path]:
    """
    Load vox2ras and vox2ras_tkr from a subject MRI volume.
    Prefer brainmask.mgz, else T1.mgz, else orig.mgz.
    """
    candidates = [
        subj_dir / "mri" / "brainmask.mgz",
        subj_dir / "mri" / "T1.mgz",
        subj_dir / "mri" / "orig.mgz",
    ]
    for p in candidates:
        if p.exists():
            img = nib.load(str(p))
            hdr = img.header
            return np.array(hdr.get_vox2ras(), float), np.array(hdr.get_vox2ras_tkr(), float), p
    raise FileNotFoundError(
        f"No MRI volume found for matrices in: {subj_dir}/mri "
        f"(brainmask.mgz/T1.mgz/orig.mgz)"
    )


def parse_talairach_xfm(xfm_path: Path) -> np.ndarray:
    """
    Parse FreeSurfer talairach.xfm and return a 4x4 affine (scannerRAS -> MNI305).
    """
    if not xfm_path.exists():
        raise FileNotFoundError(f"Missing talairach transform: {xfm_path}")

    txt = xfm_path.read_text(encoding="utf-8", errors="ignore")

    # Extract the 3x4 block between 'Linear_Transform =' and ';'
    m = re.search(r"Linear_Transform\s*=\s*([\s\S]*?);", txt)
    if not m:
        raise ValueError(f"Could not find Linear_Transform block in: {xfm_path}")

    rows = []
    for line in m.group(1).strip().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = [p for p in line.replace("\t", " ").split(" ") if p]
        if len(parts) >= 4:
            rows.append([float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])])

    if len(rows) != 3:
        raise ValueError(f"Expected 3 rows in Linear_Transform for {xfm_path}, got {len(rows)}")

    aff = np.eye(4, dtype=float)
    aff[:3, :4] = np.array(rows, dtype=float)
    return aff


def apply_affine(points_xyz: np.ndarray, aff_4x4: np.ndarray) -> np.ndarray:
    pts = np.asarray(points_xyz, float)
    hom = np.c_[pts, np.ones((pts.shape[0], 1), float)]
    out = (aff_4x4 @ hom.T).T
    return out[:, :3]


def tkr_to_scanner(points_tkr: np.ndarray, vox2ras: np.ndarray, vox2ras_tkr: np.ndarray) -> np.ndarray:
    """
    tkrRAS -> scannerRAS via voxel index space:
      voxel = inv(vox2ras_tkr) * tkr
      scanner = vox2ras * voxel
    """
    inv_vox2ras_tkr = np.linalg.inv(vox2ras_tkr)
    pts_vox = apply_affine(points_tkr, inv_vox2ras_tkr)
    pts_scanner = apply_affine(pts_vox, vox2ras)
    return pts_scanner


def scanner_to_tkr(points_scanner: np.ndarray, vox2ras: np.ndarray, vox2ras_tkr: np.ndarray) -> np.ndarray:
    """
    scannerRAS -> tkrRAS:
      voxel = inv(vox2ras) * scanner
      tkr = vox2ras_tkr * voxel
    """
    inv_vox2ras = np.linalg.inv(vox2ras)
    pts_vox = apply_affine(points_scanner, inv_vox2ras)
    pts_tkr = apply_affine(pts_vox, vox2ras_tkr)
    return pts_tkr


def subject_tkr_to_fsaverage_tkr(pid: str, points_tkr_subj: np.ndarray, fsaverage_dir: Path) -> Tuple[np.ndarray, Dict[str, str]]:
    """
    subject tkrRAS -> subject scannerRAS -> MNI305 -> fsaverage scannerRAS -> fsaverage tkrRAS
    using:
      tal_subj = subject talairach.xfm (scannerRAS -> MNI305)
      tal_fs   = fsaverage talairach.xfm (scannerRAS -> MNI305)
      inv(tal_fs): MNI305 -> fsaverage scannerRAS
    """
    subj_dir = patient_freesurfer_dir(pid)

    # Subject header + talairach
    vox2ras_subj, vox2ras_tkr_subj, subj_mgz = load_mgz_matrices(subj_dir)
    tal_subj = parse_talairach_xfm(subj_dir / "mri" / "transforms" / "talairach.xfm")

    # fsaverage header + talairach
    vox2ras_fs, vox2ras_tkr_fs, fs_mgz = load_mgz_matrices(fsaverage_dir)
    tal_fs = parse_talairach_xfm(fsaverage_dir / "mri" / "transforms" / "talairach.xfm")
    inv_tal_fs = np.linalg.inv(tal_fs)

    # subject tkr -> subject scanner
    pts_scanner_subj = tkr_to_scanner(points_tkr_subj, vox2ras_subj, vox2ras_tkr_subj)

    # subject scanner -> MNI305
    pts_mni = apply_affine(pts_scanner_subj, tal_subj)

    # MNI305 -> fsaverage scanner
    pts_scanner_fs = apply_affine(pts_mni, inv_tal_fs)

    # fsaverage scanner -> fsaverage tkr
    pts_tkr_fs = scanner_to_tkr(pts_scanner_fs, vox2ras_fs, vox2ras_tkr_fs)

    prov = {
        "pid": pid,
        "subj_dir": str(subj_dir),
        "subj_mgz_used": str(subj_mgz),
        "subj_talairach_xfm": str(subj_dir / "mri" / "transforms" / "talairach.xfm"),
        "fsaverage_dir": str(fsaverage_dir),
        "fs_mgz_used": str(fs_mgz),
        "fsaverage_talairach_xfm": str(fsaverage_dir / "mri" / "transforms" / "talairach.xfm"),
        "space_out": "fsaverage_tkrRAS_via_MNI305_affine",
    }
    return pts_tkr_fs, prov


# =============================================================================
# MAIN
# =============================================================================

def get_base_dir() -> Path:
    try:
        return Path(__file__).resolve().parent  # works in .py
    except NameError:
        return Path.cwd()  # works in Jupyter

def main() -> None:
    base_dir = get_base_dir()

    out_root = Path(OUT_ROOT)
    out_root.mkdir(parents=True, exist_ok=True)

    meta_in = base_dir / META_IN

    meta_out  = out_root / META_OUT
    coords_out = out_root / COORDS_OUT
    qc_out    = out_root / QC_OUT

    fsaverage_dir = Path(FSAVERAGE_DIR)

    print("=== RUN SUMMARY ===")
    print(f"Base dir      : {base_dir}")
    print(f"Meta in       : {meta_in}")
    print(f"Meta out      : {meta_out}")
    print(f"Coords out    : {coords_out}")
    print(f"QC out        : {qc_out}")
    print(f"Outputs root  : {OUTPUTS_ROOT}")
    print(f"FSAVERAGE_DIR : {fsaverage_dir}")
    print(f"Patients (n)  : {len(PATIENT_IDS)}")
    print("===================\n")

    # Validate fsaverage presence early
    if not fsaverage_dir.exists():
        raise FileNotFoundError(f"FSAVERAGE_DIR does not exist: {fsaverage_dir}")

    fs_tal = fsaverage_dir / "mri" / "transforms" / "talairach.xfm"
    if not fs_tal.exists():
        raise FileNotFoundError(f"Missing fsaverage talairach.xfm: {fs_tal}")

    # 1) Contact-level meta
    if not meta_in.exists():
        raise FileNotFoundError(f"Missing meta file next to script: {meta_in}")

    df_meta = read_table_auto(meta_in)
    print("[INFO] Meta columns:", list(df_meta.columns))
    df_contact = collapse_meta_to_contact_level(df_meta)
    df_contact.to_csv(meta_out, sep="\t", index=False)
    print(f"[WROTE] {meta_out}  (rows={len(df_contact)})")

    # 2) Transform patient contacts
    all_rows = []
    qc_rows = []

    for pid in PATIENT_IDS:
        try:
            subj_dir = patient_freesurfer_dir(pid)
            tal = subj_dir / "mri" / "transforms" / "talairach.xfm"
            if not tal.exists():
                raise FileNotFoundError(f"Missing subject talairach.xfm: {tal}")

            dfc = load_patient_contacts(pid)
            pts_subj = dfc[["x", "y", "z"]].to_numpy(float)

            pts_fsavg, prov = subject_tkr_to_fsaverage_tkr(pid, pts_subj, fsaverage_dir)

            out = dfc.copy()
            out["x"] = pts_fsavg[:, 0]
            out["y"] = pts_fsavg[:, 1]
            out["z"] = pts_fsavg[:, 2]
            out["coord_space"] = prov["space_out"]

            all_rows.append(out)

            qc_rows.append({
                "patient_id": pid,
                "n_contacts": int(len(out)),
                "subj_dir": prov["subj_dir"],
                "subj_mgz_used": prov["subj_mgz_used"],
                "subj_talairach_xfm": prov["subj_talairach_xfm"],
                "fsaverage_dir": prov["fsaverage_dir"],
                "fs_mgz_used": prov["fs_mgz_used"],
                "fsaverage_talairach_xfm": prov["fsaverage_talairach_xfm"],
                "space_out": prov["space_out"],
                "status": "OK",
            })

            print(f"[OK] {pid}: transformed {len(out)} contacts -> fsaverage tkrRAS")

        except Exception as e:
            qc_rows.append({
                "patient_id": pid,
                "n_contacts": 0,
                "subj_dir": str(patient_freesurfer_dir(pid)) if (str(pid).startswith("PAT_") or str(pid).startswith("MicroEPI")) else "",
                "subj_mgz_used": "",
                "subj_talairach_xfm": "",
                "fsaverage_dir": str(fsaverage_dir),
                "fs_mgz_used": "",
                "fsaverage_talairach_xfm": str(fs_tal),
                "space_out": "",
                "status": f"ERROR: {e}",
            })
            print(f"[ERROR] {pid}: {e}")

    if not all_rows:
        raise RuntimeError("No patients were transformed successfully. Check transform_qc_summary.tsv.")

    df_coords = pd.concat(all_rows, ignore_index=True)

    # Standardize minimal front columns first
    front = ["patient_id", "electrode", "x", "y", "z"]
    rest = [c for c in df_coords.columns if c not in front]
    df_coords = df_coords[front + rest]

    df_coords.to_csv(coords_out, sep="\t", index=False)
    print(f"[WROTE] {coords_out}  (rows={len(df_coords)})")

    df_qc = pd.DataFrame(qc_rows)
    df_qc.to_csv(qc_out, sep="\t", index=False)
    print(f"[WROTE] {qc_out}")

    print("\nDone.")


if __name__ == "__main__":
    main()


=== RUN SUMMARY ===
Base dir      : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\scripts
Meta in       : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\scripts\df_meta_with_clusters.tsv
Meta out      : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\atlas_inputs\df_meta_contact_level.tsv
Coords out    : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\atlas_inputs\electrode_coords_fsaverage_tkr.tsv
QC out        : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\atlas_inputs\transform_qc_summary.tsv
Outputs root  : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\01_FBM_Analysis\outputs\Paper1_recons
FSAVERAGE_DIR : \\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_HUG\fsaverage
Patients (n)  : 14

[INFO] Meta columns: ['sample_idx', 'patient_id', 'condition', 'task'

In [ ]:
#!/usr/bin/env python3
"""
01_render_atlas_AB.py

(A) Clean fsaverage pial (translucent) + depth electrodes colored by cluster
(B) DK aparc overlay (vertex RGBA) + electrodes colored by cluster

Inputs:
  - df_meta_contact_level.tsv
  - electrode_coords_fsaverage_tkr.tsv

Outputs:
  atlas_renders_AB/
    clean_brain/
      all_clusters/
      cluster_XX/
    parcellated_aparc/
      all_clusters/
      cluster_XX/
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import pyvista as pv
import matplotlib.cm as cm
import mne
from nibabel.freesurfer.io import read_geometry, read_annot


# =============================================================================
# JUPYTER-SAFE BASE DIR
# =============================================================================

def get_base_dir() -> Path:
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()


# =============================================================================
# CONFIG (WIRED TO YOUR OUTPUTS)
# =============================================================================

ATLAS_INPUTS_DIR = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\atlas_inputs")

META_FILE  = ATLAS_INPUTS_DIR / "df_meta_contact_level.tsv"
COORD_FILE = ATLAS_INPUTS_DIR / "electrode_coords_fsaverage_tkr.tsv"

OUT_DIR = ATLAS_INPUTS_DIR / "atlas_renders_AB"
SAVE_CLUSTER_COLOR_MAP = True

# Merge keys
PATIENT_COL = "patient_id"
ELECTRODE_COL = "electrode"
CLUSTER_COL = "cluster"

# Depth-only filter: if present
DEPTH_ONLY = True
SUBDURAL_COL_CANDIDATES = ["isSubdural", "subdural", "is_strip", "isGrid", "is_grid"]

# Rendering
WINDOW_SIZE = (1200, 1000)
SS_SCALE = 2
TRANSPARENT_BG = True

# Brain look (matches your pink translucent style)
BRAIN_COLOR = "#ead6db"
BRAIN_OPACITY_CLEAN = 0.35
BRAIN_OPACITY_APARC = 0.35
BRAIN_SPECULAR = 0.02
BRAIN_SPECULAR_POWER = 8
BRAIN_AMBIENT = 0.34
BRAIN_DIFFUSE = 0.66

# Electrode look
ELECTRODE_RADIUS = 1.4
ELECTRODE_OPACITY = 0.98

# Views
VIEWS_TO_SAVE = ["left", "right", "frontal", "posterior", "dorsal", "ventral"]

# Cluster plotting
PLOT_ALL_CLUSTERS = True
PLOT_ONE_CLUSTER_EACH = True

# Cluster colormap bases
CLUSTER_CMAP_BASES = ["tab20", "tab20b", "tab20c"]


# =============================================================================
# IO + MERGE
# =============================================================================

def load_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    df = pd.read_csv(path, sep="\t")
    df.columns = [c.strip() for c in df.columns]
    return df


def first_present(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    lmap = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lmap:
            return lmap[cand.lower()]
    return None


def merge_coords_and_clusters(df_meta: pd.DataFrame, df_coords: pd.DataFrame) -> pd.DataFrame:
    # normalize keys
    for df in (df_meta, df_coords):
        df[PATIENT_COL] = df[PATIENT_COL].astype(str).str.strip()
        df[ELECTRODE_COL] = df[ELECTRODE_COL].astype(str).str.strip()

    df_meta[CLUSTER_COL] = pd.to_numeric(df_meta[CLUSTER_COL], errors="coerce")
    for c in ["x", "y", "z"]:
        df_coords[c] = pd.to_numeric(df_coords[c], errors="coerce")

    merged = df_coords.merge(
        df_meta[[PATIENT_COL, ELECTRODE_COL, CLUSTER_COL]],
        how="left",
        on=[PATIENT_COL, ELECTRODE_COL],
        validate="m:1",
    )

    print("\n[QC] Merge summary")
    print(f"  - coords rows:            {len(df_coords)}")
    print(f"  - missing cluster labels: {int(merged[CLUSTER_COL].isna().sum())}")
    print(f"  - missing any x/y/z:      {int(merged[['x','y','z']].isna().any(axis=1).sum())}")

    merged = merged.dropna(subset=["x", "y", "z", CLUSTER_COL]).copy()
    merged[CLUSTER_COL] = merged[CLUSTER_COL].astype(int)

    # depth-only filter if possible
    subd = first_present(merged, SUBDURAL_COL_CANDIDATES)
    if DEPTH_ONLY and subd is not None:
        merged[subd] = pd.to_numeric(merged[subd], errors="coerce")
        before = len(merged)
        merged = merged.loc[merged[subd] == 0].copy()
        after = len(merged)
        print(f"  - DEPTH_ONLY: kept {after}/{before} using '{subd}==0'")
    elif DEPTH_ONLY:
        print("  - DEPTH_ONLY requested, but no subdural column found; skipping depth filter.")

    return merged


# =============================================================================
# COLORS
# =============================================================================

def build_cluster_color_map(cluster_labels: List[int]) -> Dict[int, Tuple[float, float, float, float]]:
    clusters = sorted(set(int(c) for c in cluster_labels))

    base_colors: List[Tuple[float, float, float, float]] = []
    for cmap_name in CLUSTER_CMAP_BASES:
        cmap = cm.get_cmap(cmap_name)
        n = getattr(cmap, "N", 20)
        for i in range(n):
            base_colors.append(cmap(i / max(1, n - 1)))

    if len(clusters) <= len(base_colors):
        colors = base_colors[:len(clusters)]
    else:
        hsv = cm.get_cmap("hsv")
        colors = [hsv(i / len(clusters)) for i in range(len(clusters))]

    return {cl: colors[i] for i, cl in enumerate(clusters)}


# =============================================================================
# FSAVERAGE LOAD + APARC
# =============================================================================

def fetch_fsaverage_dir() -> Path:
    # Use your local FSAVERAGE_DIR if present; else fall back to MNE fetch
    if ATLAS_INPUTS_DIR.exists():
        pass
    # Your provided fsaverage lives in DATARAW, but we also support MNE fallback if needed
    fs_dir = Path(r"\\nasac-m2.unige.ch\m-HumanNeuronLab\DATARAW\SEEG_EXPERIMENTS_HUG\fsaverage")
    if fs_dir.exists():
        return fs_dir
    return Path(mne.datasets.fetch_fsaverage(verbose=True))


def load_fsaverage_pial_meshes(fsaverage_dir: Path) -> Tuple[pv.PolyData, pv.PolyData]:
    surf_dir = fsaverage_dir / "surf"
    lh_path = surf_dir / "lh.pial"
    rh_path = surf_dir / "rh.pial"
    if not lh_path.exists() or not rh_path.exists():
        raise FileNotFoundError(f"Missing fsaverage pial surfaces: {lh_path} / {rh_path}")

    lh_verts, lh_faces = read_geometry(str(lh_path))
    rh_verts, rh_faces = read_geometry(str(rh_path))

    def to_pv(verts: np.ndarray, faces: np.ndarray) -> pv.PolyData:
        faces_packed = np.hstack([np.full((faces.shape[0], 1), 3, dtype=np.int64), faces]).ravel()
        mesh = pv.PolyData(verts, faces_packed)
        mesh.compute_normals(inplace=True)
        return mesh

    return to_pv(lh_verts, lh_faces), to_pv(rh_verts, rh_faces)


def build_aparc_vertex_rgba(fsaverage_dir: Path) -> Tuple[np.ndarray, np.ndarray]:
    label_dir = fsaverage_dir / "label"
    lh_annot = label_dir / "lh.aparc.annot"
    rh_annot = label_dir / "rh.aparc.annot"
    if not lh_annot.exists() or not rh_annot.exists():
        raise FileNotFoundError(f"Missing aparc annotation: {lh_annot} / {rh_annot}")

    def annot_to_rgba(annot_path: Path) -> np.ndarray:
        labels, ctab, _names = read_annot(str(annot_path))
        label_ids = ctab[:, -1]
        rgba = ctab[:, :4].astype(np.uint8)
        lut = {int(lid): rgba[i] for i, lid in enumerate(label_ids)}

        out = np.zeros((labels.shape[0], 4), dtype=np.uint8)
        default = np.array([200, 200, 200, 255], dtype=np.uint8)
        for i, lid in enumerate(labels):
            out[i] = lut.get(int(lid), default)
        return out

    return annot_to_rgba(lh_annot), annot_to_rgba(rh_annot)


# =============================================================================
# CAMERAS + RENDERING
# =============================================================================

def compute_cameras(bounds: Tuple[float, float, float, float, float, float]) -> Dict[str, Tuple[Tuple[float,float,float], Tuple[float,float,float], Tuple[float,float,float]]]:
    xmin, xmax, ymin, ymax, zmin, zmax = bounds
    cx, cy, cz = (xmin + xmax)/2, (ymin + ymax)/2, (zmin + zmax)/2
    dx, dy, dz = (xmax - xmin), (ymax - ymin), (zmax - zmin)
    size = max(dx, dy, dz)
    d = 2.4 * size

    focal = (cx, cy, cz)
    up_z = (0, 0, 1)

    return {
        "left":      ((cx - d, cy, cz), focal, up_z),
        "right":     ((cx + d, cy, cz), focal, up_z),
        "frontal":   ((cx, cy + d, cz), focal, up_z),
        "posterior": ((cx, cy - d, cz), focal, up_z),
        "dorsal":    ((cx, cy, cz + d), focal, (0, 1, 0)),
        "ventral":   ((cx, cy, cz - d), focal, (0, 1, 0)),
    }


def add_brain_mesh(plotter: pv.Plotter, mesh: pv.PolyData, rgba: Optional[np.ndarray], opacity: float) -> None:
    if rgba is None:
        plotter.add_mesh(
            mesh,
            color=BRAIN_COLOR,
            opacity=opacity,
            specular=BRAIN_SPECULAR,
            specular_power=BRAIN_SPECULAR_POWER,
            ambient=BRAIN_AMBIENT,
            diffuse=BRAIN_DIFFUSE,
            smooth_shading=True,
        )
    else:
        m = mesh.copy(deep=True)
        if rgba.shape[0] != m.n_points:
            raise ValueError("Vertex RGBA length does not match mesh vertex count.")
        m.point_data["rgba"] = rgba
        plotter.add_mesh(
            m,
            scalars="rgba",
            rgba=True,
            opacity=opacity,
            specular=BRAIN_SPECULAR,
            specular_power=BRAIN_SPECULAR_POWER,
            ambient=BRAIN_AMBIENT,
            diffuse=BRAIN_DIFFUSE,
            smooth_shading=True,
        )


def add_electrodes(plotter: pv.Plotter, df_plot: pd.DataFrame, cluster_colors: Dict[int, Tuple[float,float,float,float]]) -> None:
    for cl, dfc in df_plot.groupby(CLUSTER_COL):
        rgba = cluster_colors[int(cl)]
        rgb = rgba[:3]
        centers = dfc[["x", "y", "z"]].to_numpy(float)

        for x, y, z in centers:
            sph = pv.Sphere(radius=float(ELECTRODE_RADIUS),
                            center=(float(x), float(y), float(z)),
                            theta_resolution=18, phi_resolution=18)
            plotter.add_mesh(sph, color=rgb, opacity=ELECTRODE_OPACITY)


def render_views(
    lh: pv.PolyData,
    rh: pv.PolyData,
    df_plot: pd.DataFrame,
    cluster_colors: Dict[int, Tuple[float,float,float,float]],
    out_dir: Path,
    tag: str,
    lh_rgba: Optional[np.ndarray],
    rh_rgba: Optional[np.ndarray],
    opacity: float,
) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    bounds = (
        min(lh.bounds[0], rh.bounds[0]),
        max(lh.bounds[1], rh.bounds[1]),
        min(lh.bounds[2], rh.bounds[2]),
        max(lh.bounds[3], rh.bounds[3]),
        min(lh.bounds[4], rh.bounds[4]),
        max(lh.bounds[5], rh.bounds[5]),
    )
    cams = compute_cameras(bounds)

    for view in VIEWS_TO_SAVE:
        pl = pv.Plotter(off_screen=True, window_size=WINDOW_SIZE)
        add_brain_mesh(pl, lh, lh_rgba, opacity)
        add_brain_mesh(pl, rh, rh_rgba, opacity)
        add_electrodes(pl, df_plot, cluster_colors)
        pl.camera_position = cams[view]
        pl.reset_camera_clipping_range()

        out_png = out_dir / f"{tag}_{view}.png"
        pl.screenshot(str(out_png), transparent_background=TRANSPARENT_BG, scale=SS_SCALE)
        pl.close()
        print(f"[WROTE] {out_png}")


# =============================================================================
# MAIN
# =============================================================================

def main() -> None:
    # stability
    try:
        pv.global_theme.multi_samples = 0
    except Exception:
        pass

    print("[INFO] Loading inputs...")
    df_meta = load_table(META_FILE)
    df_coords = load_table(COORD_FILE)

    # basic column check
    for col in [PATIENT_COL, ELECTRODE_COL, CLUSTER_COL]:
        if col not in df_meta.columns:
            raise ValueError(f"Meta missing required column '{col}'. Columns: {list(df_meta.columns)}")
    for col in [PATIENT_COL, ELECTRODE_COL, "x", "y", "z"]:
        if col not in df_coords.columns:
            raise ValueError(f"Coords missing required column '{col}'. Columns: {list(df_coords.columns)}")

    df = merge_coords_and_clusters(df_meta, df_coords)
    clusters = sorted(df[CLUSTER_COL].unique().tolist())
    print(f"[INFO] Unique clusters: {len(clusters)}")

    cluster_colors = build_cluster_color_map(clusters)

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    if SAVE_CLUSTER_COLOR_MAP:
        cmap_path = OUT_DIR / "cluster_color_map.json"
        cmap_serializable = {str(k): [float(x) for x in v] for k, v in cluster_colors.items()}
        cmap_path.write_text(json.dumps(cmap_serializable, indent=2), encoding="utf-8")
        print(f"[WROTE] {cmap_path}")

    print("[INFO] Loading fsaverage meshes + aparc...")
    fs_dir = fetch_fsaverage_dir()
    lh, rh = load_fsaverage_pial_meshes(fs_dir)
    lh_aparc_rgba, rh_aparc_rgba = build_aparc_vertex_rgba(fs_dir)

    # A) Clean brain
    if PLOT_ALL_CLUSTERS:
        render_views(
            lh, rh, df, cluster_colors,
            out_dir=OUT_DIR / "clean_brain" / "all_clusters",
            tag="clean_allclusters",
            lh_rgba=None, rh_rgba=None,
            opacity=BRAIN_OPACITY_CLEAN,
        )

    if PLOT_ONE_CLUSTER_EACH:
        for cl in clusters:
            dfc = df.loc[df[CLUSTER_COL] == cl].copy()
            render_views(
                lh, rh, dfc, cluster_colors,
                out_dir=OUT_DIR / "clean_brain" / f"cluster_{int(cl):02d}",
                tag=f"clean_cluster_{int(cl):02d}",
                lh_rgba=None, rh_rgba=None,
                opacity=BRAIN_OPACITY_CLEAN,
            )

    # B) aparc overlay
    if PLOT_ALL_CLUSTERS:
        render_views(
            lh, rh, df, cluster_colors,
            out_dir=OUT_DIR / "parcellated_aparc" / "all_clusters",
            tag="aparc_allclusters",
            lh_rgba=lh_aparc_rgba, rh_rgba=rh_aparc_rgba,
            opacity=BRAIN_OPACITY_APARC,
        )

    if PLOT_ONE_CLUSTER_EACH:
        for cl in clusters:
            dfc = df.loc[df[CLUSTER_COL] == cl].copy()
            render_views(
                lh, rh, dfc, cluster_colors,
                out_dir=OUT_DIR / "parcellated_aparc" / f"cluster_{int(cl):02d}",
                tag=f"aparc_cluster_{int(cl):02d}",
                lh_rgba=lh_aparc_rgba, rh_rgba=rh_aparc_rgba,
                opacity=BRAIN_OPACITY_APARC,
            )

    print("\n[INFO] Done.")
    print(f"[INFO] Outputs: {OUT_DIR}")


if __name__ == "__main__":
    main()


[INFO] Loading inputs...

[QC] Merge summary
  - coords rows:            1277
  - missing cluster labels: 1183
  - missing any x/y/z:      0
  - DEPTH_ONLY: kept 68/94 using 'isSubdural==0'
[INFO] Unique clusters: 10
[WROTE] \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\atlas_inputs\atlas_renders_AB\cluster_color_map.json
[INFO] Loading fsaverage meshes + aparc...


C:\Users\artoni\AppData\Local\Temp\ipykernel_26948\218221030.py:165: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = cm.get_cmap(cmap_name)


[WROTE] \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\atlas_inputs\atlas_renders_AB\clean_brain\all_clusters\clean_allclusters_left.png
[WROTE] \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\atlas_inputs\atlas_renders_AB\clean_brain\all_clusters\clean_allclusters_right.png
